# tACS Bandit: Main Analyses

**Study:** Examining the effects of theta-tACS over left DLPFC on reward-based learning across the adult lifespan

**Design:** Within-subject crossover (active vs. sham theta-tACS, 6 Hz)

This notebook orchestrates all pre-registered and exploratory analyses by calling modular functions. Individual modules contain the implementation details.

---
## 0. Setup & Imports

In [ ]:
# Standard
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_white'

# Project modules
from config import *
from data_loading import load_all_subjects, summarize_loading
from exclusions import apply_all_exclusions
from sample_descriptives import run_sample_descriptives
from blinding_analysis import run_blinding_analysis
from wsls import run_wsls_analysis
from reversal_analysis import run_reversal_analysis
from rescorla_wagner import run_rw_analysis
from ddm import run_ddm_analysis
from eeg_theta import run_theta_analysis
from cognitive_merge import run_cognitive_merge
from hypothesis_tests import run_hypothesis_tests
from order_effects import run_order_effects_analysis
from correlations import run_correlation_analysis

print(f'Study: {len(SUBJECT_INFO)} participants in SUBJECT_INFO')
print(f'Data directory: {DATA_DIR}')

---
## 1. Data Loading & Preprocessing

In [ ]:
# Load all behavioral data
data = load_all_subjects()
summarize_loading(data)

In [ ]:
# Apply exclusion criteria
exclusion_results = apply_all_exclusions(data)

# Extract filtered datasets
data_h1 = exclusion_results['data_h1']       # H1 analyses: sham only, behavioral exclusions
data_h2 = exclusion_results['data_h2']       # H2 analyses: active vs sham, all exclusions
data_clean = exclusion_results['data_clean'] # All clean trials
run_exclusions = exclusion_results['run_exclusions']
h2_eligible = exclusion_results['h2_eligible']

print(f'\nH1-eligible subjects: {data_h1["subject_id"].nunique()}')
print(f'H2-eligible subjects: {len(h2_eligible)}')

---
## 2. Sample Descriptives & Demographics

In [ ]:
# Demographics and cap size analysis
desc_results = run_sample_descriptives()

---
## 3. Blinding Integrity

In [ ]:
# Signal detection analysis of blinding
blinding_results = run_blinding_analysis(data_clean)

if blinding_results['blinding_intact']:
    print('\n✓ Blinding appears intact')
else:
    print('\n⚠ Blinding may be compromised — interpret with caution')

---
## 4. Win-Stay / Lose-Shift Analysis

In [ ]:
# WSLS analysis for H1 and H2
wsls_results = run_wsls_analysis(data_h1, data_h2, data)

---
## 5. Reversal Learning Analysis

In [ ]:
# Reversal-locked accuracy and trials to criterion
rev_results = run_reversal_analysis(data, data_clean, conditions=['sham', 'active'])

---
## 6. Rescorla-Wagner Model

In [ ]:
# Fit R-W model (MLE and MAP)
rw_results = run_rw_analysis(
    data_clean, 
    h2_subjects=h2_eligible,
    run_recovery=False  # Set True to run parameter recovery (slow)
)

---
## 7. Drift Diffusion Model

In [ ]:
# Fit DDM (requires pyddm)
ddm_results = run_ddm_analysis(
    data_clean,
    conditions=['sham', 'active'],
    run_bootstrap=False,  # Set True for bootstrap CI (slow)
    n_bootstrap=200
)

---
## 8. EEG Theta Reactivity

In [ ]:
# Compute theta reactivity from baseline EEG (F4 channel)
# Note: Requires EEG .easy files in DATA_DIR/nic/raw
try:
    theta_results = run_theta_analysis(verbose=True, show_plots=True)
    theta_subject = theta_results.get('theta_subject')
    print(f'\nTheta reliability (ICC): {theta_results["reliability"]["ICC"]:.3f}')
except FileNotFoundError:
    print('EEG files not found - skipping theta analysis')
    theta_results = None
    theta_subject = None

---
## 9. Cognitive & Survey Data Integration

In [ ]:
# Load external data and build subject DataFrame
cog_results = run_cognitive_merge(
    wsls_h1=wsls_results.get('wsls_h1'),
    wsls_h2=wsls_results.get('wsls_h2'),
    rw_mle=rw_results.get('rw_mle'),
    ddm_params=ddm_results.get('ddm_subject') if ddm_results else None,
    theta_subject=theta_subject,
    h2_subjects=h2_eligible
)

subj_df = cog_results['subj_df']
print(f'\nSubject DataFrame: {len(subj_df)} subjects, {len(subj_df.columns)} variables')
if 'theta_p95' in subj_df.columns:
    print(f'Subjects with theta data: {subj_df["theta_p95"].notna().sum()}')

---
## 10. Pre-Registered Hypothesis Tests

In [ ]:
# Run all H1 and H2 tests
hyp_results = run_hypothesis_tests(subj_df, show_plots=True)

---
## 11. Order & Session Effects

In [ ]:
# Check for counterbalance confounds
order_results = run_order_effects_analysis(subj_df)

---
## 12. Correlation Matrix

In [ ]:
# Pairwise correlations among pre-registered variables
# (now includes theta_p95)
corr_results = run_correlation_analysis(subj_df, include_tertiary=False)

---
## 13. Summary

In [ ]:
print('='*70)
print('ANALYSIS SUMMARY')
print('='*70)
print()

# Sample
print(f'Sample: N = {len(subj_df)}')
print(f'  H1-eligible (sham): {data_h1["subject_id"].nunique()}')
print(f'  H2-eligible (paired): {len(h2_eligible)}')
if 'theta_p95' in subj_df.columns:
    print(f'  With theta data: {subj_df["theta_p95"].notna().sum()}')
print()

# Blinding
if blinding_results['blinding_intact']:
    print('✓ Blinding intact')
else:
    print('⚠ Blinding concern')

# Order effects
if order_results['any_significant_order']:
    print('⚠ Significant order × condition interaction')
else:
    print('✓ No significant order effects')

# Theta reliability
if theta_results is not None and 'reliability' in theta_results:
    icc = theta_results['reliability'].get('ICC', np.nan)
    if not np.isnan(icc):
        if icc >= 0.75:
            print(f'✓ Theta reliability excellent (ICC = {icc:.2f})')
        elif icc >= 0.60:
            print(f'✓ Theta reliability good (ICC = {icc:.2f})')
        else:
            print(f'⚠ Theta reliability fair/poor (ICC = {icc:.2f})')
print()

# H2.1 summary
print('H2.1 Paired Comparisons (tACS vs. Sham):')
for param, res in hyp_results['h2_1'].items():
    if res is not None:
        sig = '*' if res['p'] < 0.05 else ''
        print(f"  {param}: dz = {res['dz']:.3f}, p = {res['p']:.3f} {sig}")

---
## Module Reference

| Module | Purpose |
|--------|--------|
| `config.py` | Paths, SUBJECT_INFO, colors, constants |
| `data_loading.py` | Load and preprocess behavioral data |
| `exclusions.py` | Apply behavioral and stim exclusion criteria |
| `sample_descriptives.py` | Demographics, age distribution, cap size |
| `blinding_analysis.py` | Signal detection analysis of blinding |
| `wsls.py` | Win-stay/lose-shift analysis |
| `reversal_analysis.py` | Reversal-locked accuracy curves |
| `rescorla_wagner.py` | R-W model fitting (MLE/MAP) |
| `ddm.py` | Drift diffusion model fitting |
| `eeg_theta.py` | Theta reactivity from baseline EEG |
| `cognitive_merge.py` | External data integration, composites |
| `hypothesis_tests.py` | H1.1, H1.2, H2.1, H2.2, theta moderation |
| `order_effects.py` | Counterbalance and session effects |
| `correlations.py` | Correlation heatmap |
| `plotting_utils.py` | Shared visualization utilities |